In [1]:
# import sys
import glob
from pathlib import Path
import os
import zarr

import time
import gc
import numpy as np
import xarray as xr
# import pandas as pd
# import calendar
import dask.array as da
# from scipy.interpolate import griddata
from scipy.spatial import Delaunay
from scipy.interpolate import LinearNDInterpolator

import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
import cartopy.crs as ccrs
import cmocean # https://matplotlib.org/cmocean/
# import pop_tools # https://pop-tools.readthedocs.io/en/latest/
# import gsw

### 'sys.path.insert' adds our CESM2 data handling tools to the global namespace
### make sure you are referencing the path in which you downloaded the tools
# sys.path.insert(0, "/home/prouenhoff/ondemand/cesm_tools/cesm_tools/")
# import cesm_scenarios
# import geo_tools
# import pop_plot

In [2]:
assert False

AssertionError: 

### User input

In [3]:
### Choose which variable to process via var: HMXL for mixed layer depth, TREFHT for surface temperature, snow_prec for snow precipitation, liq_prec for liquid precipitation.
### The upper case variable names are also used by CESM, but for precipitation it's different (snow precip rate = PRECSC + PRECSL, liquid precip rate = PRECT - snow...)

var  = "HMXL"

In [4]:
### For each possible variable, we have to do similar things:
### preprocess in order to at least get a DataArray instead of a Dataset,
### convert the original unit into a more sensible one (for HMXL from cm -> m, for any precip from m/s -> mm/d, TREFHT K -> °C),
### provide that unit as a string for nice plotting,
### as well as the module (ocn or atm, maybe lnd in the future) to get paths via another dict (see below)

vars_dict = {
    "HMXL": {
        "preprocess": lambda ds: ds['HMXL'],
        "convert_function": lambda da: da / 100.0, # cm -> m
        "unit": "[m]",
        "module": "ocn",
    },
    "TREFHT": {
        "preprocess": lambda ds: ds['TREFHT'],
        "convert_function": lambda da: da - 273.15, # K -> °C
        "unit": "[°C]",
        "module": "atm",
    },
    "PminusE": {
        "preprocess": lambda ds: (ds['PRECT'] - ds['PRECSC'] - ds['PRECSL'] - evaprate_from_lhflx(ds['TREFHT'],ds['LHFLX'])).rename("PminusE"),
        "convert_function": lambda da: da * 1000 * 86400,   # m/s -> mm/d
        "unit": "[mm/d]",
        "module": "atm",
    },
}

def evaprate_from_lhflx(temp, lhflx):
    vaporizing_enthalpy = (2500.9 - 2.3989 * temp) * 1e3  # unit: J/kg, source: Asensio et al., 2023
    evaprate = lhflx / vaporizing_enthalpy # W/m2 / (J/kg) = J/s/m2/J*kg = kg/s/m2 = mm/s = m/s 1e-3
    return evaprate * 1e-3


### A dict holding the paths and filenames depending on which module (lnd or ocn) the variable of choice is from
path_dict = {
    "ocn": {
        "path_lr": "/projects/0/nwo2021025/archive/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050_twin_no_sai.001/ocn/hist/",
        "path_hr": "/projects/0/uuesm2/cesm1_0_4/f05_t12/extracted/rcp8.5_co2_f05_t12/ocn/",
        "path_lr_SAI": "/projects/0/nwo2021025/archive/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050.001/ocn/hist/",
        "path_hr_SAI": "/projects/0/nwo2021025/archive/mres_b.e10.B2000_CAM5.f05_t12.001/ocn/hist/",
        "file_name_lr": "lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050_twin_no_sai.001.pop.h.2*.nc",
        "file_name_hr": "POP_subset_2*.nc",
        "file_name_lr_SAI": "lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050.001.pop.h.2*.nc",
        "file_name_hr_SAI": "mres_b.e10.B2000_CAM5.f05_t12.001.pop.h.2*.nc",
    },
    "atm": {
        "path_lr": "/projects/0/nwo2021025/archive/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050_twin_no_sai.001/atm/hist/",
        "path_hr": "/projects/0/uuesm2/cesm1_0_4/f05_t12/extracted/rcp8.5_co2_f05_t12/atm/",
        "path_lr_SAI": "/projects/0/nwo2021025/archive/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050.001/atm/hist/",
        "path_hr_SAI": "/projects/0/nwo2021025/archive/mres_b.e10.B2000_CAM5.f05_t12.001/atm/hist/",
        "file_name_lr": "lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050_twin_no_sai.001.cam2.h0.2*.nc",
        "file_name_hr": "CAM_subset_2*.nc",
        "file_name_lr_SAI": "lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050.001.cam2.h0.2*.nc",
        "file_name_hr_SAI": "mres_b.e10.B2000_CAM5.f05_t12.001.cam2.h0.2*.nc",
    }
}

if var not in vars_dict:
    raise Exception(f"The variable \"{var}\" is not in the list of designated variables, see vars_dict")

### Load raw data and post-process

In [5]:
### We want to compare low-resolution and high-resoultion simulation data, so we create _lr and _hr variables (with and without SAI)
module = vars_dict[var]["module"]

path_lr, path_hr, file_name_lr, file_name_hr = path_dict[module]['path_lr'], path_dict[module]['path_hr'], path_dict[module]['file_name_lr'], path_dict[module]['file_name_hr']

file_list_lr = glob.glob(path_lr + file_name_lr)
file_list_hr = glob.glob(path_hr + file_name_hr)
file_list_lr.sort()
file_list_hr.sort()

# del file_list_hr[0:50*12]
# del file_list_hr[-12:]
del file_list_lr[-12:]
print(file_list_hr[0], "\n ", file_list_lr[0], "\n ", file_list_hr[-1], "\n ", file_list_lr[-1])

path_lr_SAI, path_hr_SAI, file_name_lr_SAI, file_name_hr_SAI = path_dict[module]['path_lr_SAI'], path_dict[module]['path_hr_SAI'], path_dict[module]['file_name_lr_SAI'], path_dict[module]['file_name_hr_SAI']

file_list_lr_SAI = glob.glob(path_lr_SAI + file_name_lr_SAI)
file_list_hr_SAI = glob.glob(path_hr_SAI + file_name_hr_SAI)
file_list_lr_SAI.sort()
file_list_hr_SAI.sort()

# del file_list_lr_SAI[0:45*12]
print("\n")
print(file_list_hr_SAI[0], "\n ", file_list_lr_SAI[0], "\n ", file_list_hr_SAI[-1], "\n ", file_list_lr_SAI[-1])

/projects/0/uuesm2/cesm1_0_4/f05_t12/extracted/rcp8.5_co2_f05_t12/ocn/POP_subset_2000-01.nc 
  /projects/0/nwo2021025/archive/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050_twin_no_sai.001/ocn/hist/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050_twin_no_sai.001.pop.h.2050-01.nc 
  /projects/0/uuesm2/cesm1_0_4/f05_t12/extracted/rcp8.5_co2_f05_t12/ocn/POP_subset_2100-12.nc 
  /projects/0/nwo2021025/archive/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050_twin_no_sai.001/ocn/hist/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050_twin_no_sai.001.pop.h.2099-12.nc


/projects/0/nwo2021025/archive/mres_b.e10.B2000_CAM5.f05_t12.001/ocn/hist/mres_b.e10.B2000_CAM5.f05_t12.001.pop.h.2045-01.nc 
  /projects/0/nwo2021025/archive/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050.001/ocn/hist/lres_b.e10.B2000_CAM5.f09_g16.feedforward_2050.001.pop.h.2000-01.nc 
  /projects/0/nwo2021025/archive/mres_b.e10.B2000_CAM5.f05_t12.001/ocn/hist/mres_b.e10.B2000_CAM5.f05_t12.001.pop.h.2098-12.nc 
  /projects/0/nwo202102

In [19]:
num_months_to_load = -1

# path_post_processed = "C://Users//proue//Downloads//post_processed/"
path_post_processed = "/home/prouenhoff/ondemand/resol_comparison/" + var + "/post_processed/"
path_noSAI_hr = path_post_processed + var + "_noSAI_hr.zarr" #/home/prouenhoff/ondemand/resol_comparison/" + var + "/post_processed/" + var + "_noSAI_hr.zarr"
path_noSAI_lr = path_post_processed + var + "_noSAI_lr.zarr" #/home/prouenhoff/ondemand/resol_comparison/" + var + "/post_processed/" + var + "_noSAI_lr.zarr"



if os.path.exists(path_noSAI_lr):
    data_lr = xr.open_zarr(path_noSAI_lr)[var]
    # assert data_lr.name == var, f"DataArray is called {data_lr.name}, expected {var}"
else:
    print("Loading data_lr")
    time_start = time.perf_counter()

    os.makedirs(path_post_processed, exist_ok=True)

    preprocess = vars_dict[var]["preprocess"]
    
    data_lr = xr.open_mfdataset(
        file_list_lr[0:num_months_to_load],
        preprocess=preprocess,
        combine="by_coords",
        data_vars="minimal",    # only take over new variables from first file
        coords="minimal",       # no duplication of coordinates
        compat="override",      # ignore small coordinate inconsistencies
        decode_timedelta=False, # because of some future compatibility warning...
        chunks={"time": 1},     # so data_lr.isel(time=0) only loads a chunk of this size
        # parallel=False         # if set to True for data_hr below, kernel will die :(
    )

    # assert data_lr.name == var, f"DataArray is called {data_lr.name}, expected {var}"

    time_end = time.perf_counter()
    print(f"Load and concat time for {data_lr.sizes['time']} months: {(time_end - time_start)/60:.1f} minutes")

    data_lr.to_zarr(
        path_noSAI_lr, 
        mode="w-",
    )



if os.path.exists(path_noSAI_hr):
    data_hr = xr.open_zarr(path_noSAI_hr)[var]
    # assert data_hr.name == var, f"DataArray is called {data_lr.name}, expected {var}"

else:
    print("Loading data_hr")
    time_start = time.perf_counter()

    os.makedirs(path_post_processed, exist_ok=True)

    preprocess = vars_dict[var]["preprocess"]

    data_hr = xr.open_mfdataset(
        file_list_hr[0:num_months_to_load],
        preprocess=preprocess,
        combine="by_coords",
        data_vars="minimal",
        coords="minimal",
        compat="override",
        decode_timedelta=False,
        chunks={"time":1},
        # parallel=False
    )

    # assert data_hr.name == var, f"DataArray is called {data_lr.name}, expected {var}"

    time_end = time.perf_counter()
    print(f"Load and concat time for {data_hr.sizes['time']} months: {(time_end - time_start)/60:.1f} minutes")

    data_hr.to_zarr(
        path_noSAI_hr,
        mode="w-",
    )

In [20]:
path_SAI_hr = path_post_processed + var + "_SAI_hr.zarr"
path_SAI_lr = path_post_processed + var + "_SAI_lr.zarr"



if os.path.exists(path_SAI_lr):
    data_lr_SAI = xr.open_zarr(path_SAI_lr)[var]
    # assert data_lr_SAI.name == var, f"DataArray is called {data_lr.name}, expected {var}"
else:
    print("Loading data_lr_SAI")
    time_start = time.perf_counter()

    os.makedirs(path_post_processed, exist_ok=True)

    preprocess = vars_dict[var]["preprocess"]
    
    data_lr_SAI = xr.open_mfdataset(
        file_list_lr_SAI[0:num_months_to_load],
        preprocess=preprocess,
        combine="by_coords",
        data_vars="minimal",    # only take over new variables from first file
        coords="minimal",       # no duplication of coordinates
        compat="override",      # ignore small coordinate inconsistencies
        decode_timedelta=False, # because of some future compatibility warning...
        chunks={"time": 1},     # so data_lr.isel(time=0) only loads a chunk of this size
        # parallel=False         # if set to True for data_hr below, kernel will die :(
    )

    # assert data_lr_SAI.name == var, f"DataArray is called {data_lr.name}, expected {var}"

    time_end = time.perf_counter()
    print(f"Load and concat time for {data_lr_SAI.sizes['time']} months: {(time_end - time_start)/60:.1f} minutes")

    data_lr_SAI.to_zarr(
        path_SAI_lr, 
        mode="w-",
    )



if os.path.exists(path_SAI_hr):
    data_hr_SAI = xr.open_zarr(path_SAI_hr)[var]
    # assert data_hr_SAI.name == var, f"DataArray is called {data_lr.name}, expected {var}"
else:
    print("Loading data_hr_SAI")
    time_start = time.perf_counter()

    os.makedirs(path_post_processed, exist_ok=True)

    preprocess = vars_dict[var]["preprocess"]

    data_hr_SAI = xr.open_mfdataset(
        file_list_hr_SAI[0:num_months_to_load],
        preprocess=preprocess,
        combine="by_coords",
        data_vars="minimal",
        coords="minimal",
        compat="override",
        decode_timedelta=False,
        chunks={"time":1},
        # parallel=False
    )

    # assert data_hr_SAI.name == var, f"DataArray is called {data_lr.name}, expected {var}"

    time_end = time.perf_counter()
    print(f"Load and concat time for {data_hr_SAI.sizes['time']} months: {(time_end - time_start)/60:.1f} minutes")

    data_hr_SAI.to_zarr(
        path_SAI_hr,
        mode="w-",
    )

In [21]:
### Convert the data into the right units
convert_func = vars_dict[var]["convert_function"]

data_hr = convert_func(data_hr)
data_lr = convert_func(data_lr)
data_hr_SAI = convert_func(data_hr_SAI)
data_lr_SAI = convert_func(data_lr_SAI)

# if var == "HMXL":
#     print("HMXL gets some special treatment")
#     data_hr = data_hr.where(data_hr != -1.)
    # data_hr = data_hr.reset_coords("z_t", drop=True)    # data_lr doesn't have that

In [22]:
### We define a common plotting range via the (more restricted) range of the HR data
# TLONG_min = data_hr["TLONG"].min().compute().item() # -109.49889346837136
# TLONG_max = data_hr["TLONG"].max().compute().item() # 69.49889346837138
# TLAT_min = data_hr["TLAT"].where(data_hr["TLAT"] != -1.0 ).min().compute().item() # 0.05000001903858511
# TLAT_max = data_hr["TLAT"].max().compute().item() # 89.58694891134195
# print(TLONG_min, ", ", TLONG_max, ", ", TLAT_min, ", ", TLAT_max)
TLONG_min = -81.0
TLONG_max = 25.0
TLAT_min = 36.0
# TLAT_max = data_hr["TLAT"].max().compute().item() # 89.58694891134195
TLAT_max = 89.58694891134195

### Interpolate in order to build difference across different resolutions

In [ ]:
example_ds_lr = data_lr.sel(time="2050-02-01").squeeze(dim="time")
example_ds_hr = data_hr.sel(time="2050-02-01").squeeze(dim="time")

data_noSAI_interpol_path = "/home/prouenhoff/ondemand/resol_comparison/" + var + "/post_processed/" + var + "_noSAI_interpol_raw.zarr"

if os.path.exists(data_noSAI_interpol_path):
    data_lr_on_hr = xr.DataArray(
        da.from_zarr(data_noSAI_interpol_path), # stay lazy
        dims=["time", "nlat", "nlon"],
        coords={
            "time": data_lr.time,
            "TLONG": example_ds_hr.TLONG.load(),
            "TLAT": example_ds_hr.TLAT.load(),
            },
        )

else:
    ### Get the physical unregular grid coordinates
    tlong_lr = example_ds_lr.TLONG.values
    tlat_lr = example_ds_lr.TLAT.values

    tlong_hr = example_ds_hr.TLONG.values
    tlong_hr[tlong_hr != -1] = tlong_hr[tlong_hr != -1] %360
    tlat_hr = example_ds_hr.TLAT.values

    lr_points = np.column_stack([tlong_lr.ravel(), tlat_lr.ravel()])
    hr_points = np.column_stack([tlong_hr.ravel(), tlat_hr.ravel()])

    ### Interpolate one time step at a time
    time_start = time.perf_counter()
    print("Started timer")

    ### Triangulate once
    tri = Delaunay(lr_points)
    interpolator = LinearNDInterpolator(tri, values= np.zeros(len(lr_points)))

    result = zarr.open(
        data_noSAI_interpol_path,
        mode="w-",
        shape=(data_lr.sizes["time"], *tlong_hr.shape),
        dtype=np.float32,
        chunks=(1, *tlong_hr.shape),
        fill_value=np.nan
    )

    ### Interpolate for each time
    for t in range(data_lr.sizes["time"]):
        if t%100 == 1:
            print(t)
        vals = data_lr.isel(time=t).load().values.ravel().astype(np.float64)
        interpolator.values = vals.reshape(-1, 1)
        result[t] = interpolator(hr_points).reshape(tlong_hr.shape).astype(np.float32)
        del vals
        gc.collect()

    # result.close()

    data_lr_on_hr = xr.DataArray(
    da.from_zarr(data_noSAI_interpol_path), # stay lazy
    dims=["time", "nlat", "nlon"],
    coords={
        "time": data_lr.time,
        "TLONG": example_ds_hr.TLONG.load(),
        "TLAT": example_ds_hr.TLAT.load(),
        },
    )
    time_end = time.perf_counter()
    print(f"Interpolation time with {data_hr.sizes['time']} months: {(time_end - time_start)/60:.1f} minutes") # approx. 0.7 seconds per month

In [ ]:
example_ds_lr_SAI = data_lr_SAI.sel(time="2045-02-01").squeeze(dim="time")
example_ds_hr_SAI = data_hr_SAI.sel(time="2045-02-01").squeeze(dim="time")

data_SAI_interpol_path = "/home/prouenhoff/ondemand/resol_comparison/" + var + "/post_processed/" + var + "_SAI_interpol_raw.zarr"

if os.path.exists(data_SAI_interpol_path):
    data_lr_on_hr_SAI = xr.DataArray(
        da.from_zarr(data_SAI_interpol_path), # stay lazy
        dims=["time", "nlat", "nlon"],
        coords={
            "time": data_hr_SAI.time,
            "TLONG": example_ds_hr_SAI.TLONG.load(),
            "TLAT": example_ds_hr_SAI.TLAT.load(),
            },
        )

else:
    ### Get the physical unregular grid coordinates
    tlong_lr = example_ds_lr_SAI.TLONG.values
    tlat_lr = example_ds_lr_SAI.TLAT.values

    tlong_hr = example_ds_hr_SAI.TLONG.values
    tlong_hr[tlong_hr != -1] = tlong_hr[tlong_hr != -1] %360
    tlat_hr = example_ds_hr_SAI.TLAT.values

    lr_points = np.column_stack([tlong_lr.ravel(), tlat_lr.ravel()])
    hr_points = np.column_stack([tlong_hr.ravel(), tlat_hr.ravel()])

    ### Interpolate one time step at a time
    time_start = time.perf_counter()
    print("Started timer")

    ### Triangulate once
    tri = Delaunay(lr_points)
    interpolator = LinearNDInterpolator(tri, values= np.zeros(len(lr_points)))

    result = zarr.open(
        data_SAI_interpol_path,
        mode="w-",
        shape=(data_lr_SAI.sizes["time"], *tlong_hr.shape),
        dtype=np.float32,
        chunks=(1, *tlong_hr.shape),
        fill_value=np.nan
    )

    ### Interpolate for each time
    for t in range(data_lr_SAI.sizes["time"]):
        if t%100 == 1:
            print(t)
        vals = data_lr_SAI.isel(time=t).load().values.ravel().astype(np.float64)
        interpolator.values = vals.reshape(-1, 1)
        result[t] = interpolator(hr_points).reshape(tlong_hr.shape).astype(np.float32)
        del vals
        gc.collect()

    result.close()

    data_lr_on_hr_SAI = xr.DataArray(
    da.from_zarr(data_SAI_interpol_path), # stay lazy
    dims=["time", "nlat", "nlon"],
    coords={
        "time": data_lr_SAI.time,
        "TLONG": example_ds_hr_SAI.TLONG.load(),
        "TLAT": example_ds_hr_SAI.TLAT.load(),
        },
    )
    time_end = time.perf_counter()
    print(f"Interpolation time with {data_hr_SAI.sizes['time']} months: {(time_end - time_start)/60:.1f} minutes") # approx. 0.7 seconds per month

In [13]:
# ### To subtract data_hr from data_hr_SAI, we need to cut out the Atlantic from data_hr_SAI, as was already done in data_hr
# ### First, shift TLONG from data_hr_SAI from 0...360 to -180...180
# new_hr_SAI_TLONG = ((data_hr_SAI.TLONG + 180) % 360) - 180
# data_hr_SAI = data_hr_SAI.assign_coords(TLONG=new_hr_SAI_TLONG)

# ### TLAT[0,0] and TLONG[0,0] are different for both! Find nlat0, nlon0 s.t. TLAT[0,0] = SAI_TLAT[nlat0, nlon0] etc.:
# tlong0 = data_hr.TLONG[0,0].values.item()
# tlat0 = data_hr.TLAT[0,0].values.item()
# dlon = data_hr_SAI.TLONG.values - tlong0
# dlat = data_hr_SAI.TLAT.values - tlat0
# index0 = np.argmin(dlon**2 + dlat**2)
# nlat0, nlon0 = np.unravel_index(index0, (len(data_hr_SAI.nlat), len(data_hr_SAI.nlon)))

# ### In order for the indices to be aligned, first roll the SAI data such that TLAT[0,0] is the same for both sets, then cut via .isel()
# data_hr_SAI = data_hr_SAI.roll(nlat = -nlat0, nlon = -nlon0, roll_coords = True).isel(nlat = slice(0, data_hr.nlat.size), nlon = slice(0, data_hr.nlon.size))
# data_lr_on_hr_SAI = data_lr_on_hr_SAI.roll(nlat = -nlat0, nlon = -nlon0, roll_coords = True).isel(nlat = slice(0, data_hr.nlat.size), nlon = slice(0, data_hr.nlon.size))

NameError: name 'data_lr_on_hr_SAI' is not defined

In [ ]:
# data_diff_noSAI = data_hr - data_lr_on_hr
# data_diff_noSAI = data_diff_noSAI.assign_coords(TLONG=data_hr.TLONG)
# data_diff_noSAI = data_diff_noSAI.assign_coords(TLAT=data_hr.TLAT)

# data_diff_SAI = data_hr_SAI - data_lr_on_hr_SAI
# data_diff_SAI = data_diff_SAI.assign_coords(TLONG=data_hr_SAI.TLONG)
# data_diff_SAI = data_diff_SAI.assign_coords(TLAT=data_hr_SAI.TLAT)

data_diff_hr = data_hr_SAI - data_hr
data_diff_hr = data_diff_hr.assign_coords(TLONG=data_hr.TLONG)
data_diff_hr = data_diff_hr.assign_coords(TLAT=data_hr.TLAT)

data_diff_lr = data_lr_SAI - data_lr
data_diff_lr = data_diff_lr.assign_coords(TLONG=data_lr.TLONG)
data_diff_lr = data_diff_lr.assign_coords(TLAT=data_lr.TLAT)

# Plotting the data

In [14]:
### Ranges for the color bars
data2D_min = 0.0
if var != "HMXL":
    data2D_min = np.min(
        [np.nanmin(data_lr.data).compute().item(),
         np.nanmin(data_hr.data).compute().item(),
         np.nanmin(data_lr_SAI.data).compute().item(),
         np.nanmin(data_hr_SAI.data).compute().item()]
    )

data2D_max = np.max(
    [np.nanmax(data_lr.data).compute().item(),
     np.nanmax(data_hr.data).compute().item(),
     np.nanmax(data_lr_SAI.data).compute().item(),
     np.nanmax(data_hr_SAI.data).compute().item()]
)
# diff_absmax = np.max(
#     [#np.nanmax(np.abs(data_diff.data)).compute(),
#      np.nanmax(np.abs(data_diff_SAI.data)).compute(),
#      np.nanmax(np.abs(data_diff_hr.data)).compute(),]
# )
# diff_min = -diff_absmax
# diff_max = diff_absmax

print(data2D_min, ", ", data2D_max)#, ", ", diff_min, ", ", diff_max)

0.0 ,  2532.011962890625


In [ ]:
# import matplotlib.pyplot as plt
# from matplotlib.gridspec import GridSpec
# import cmocean
# import cartopy.crs as ccrs
#
# def plot_2D_resolcomp(
#     data_highres,
#     data_lowres,
#     data_difference,
#     title,
#     minval_forcolorbar=data2D_min,
#     maxval_forcolorbar=data2D_max,
#     absmaxval_forcolorbar=diff_absmax,
#     lon_west=TLONG_min,
#     lon_east=TLONG_max,
#     lat_south=TLAT_min,
#     lat_north=TLAT_max,
# ):
#     fig = plt.figure(figsize=(15, 5))
#
#     proj = ccrs.Robinson(central_longitude=(lon_west + lon_east) / 2)
#     tran = ccrs.PlateCarree()
#     ax1 = fig.add_subplot(1, 3, 1, projection=proj)
#     ax2 = fig.add_subplot(1, 3, 2, projection=proj)
#     ax3 = fig.add_subplot(1, 3, 3, projection=proj)
#
#     for ax in [ax1, ax2, ax3]:
#         # ax.gridlines()
#         ax.coastlines(resolution='110m', color='black')
#         ax.set_extent((lon_west, lon_east, lat_south, lat_north))
#         # if isinstance(proj, ccrs.Robinson):
#         #     ax.set_facecolor("white")
#
#     p1 = data_highres.plot(
#         ax=ax1,
#         x="TLONG",
#         y="TLAT",
#         transform=tran,
#         cmap=cmocean.cm.deep,
#         vmin=minval_forcolorbar,
#         vmax=maxval_forcolorbar,
#         add_colorbar=False
#     )
#     p2 = data_lowres.plot(
#         ax=ax2,
#         x="TLONG",
#         y="TLAT",
#         transform=tran,
#         cmap=cmocean.cm.deep,
#         vmin=minval_forcolorbar,
#         vmax=maxval_forcolorbar,
#         add_colorbar=False
#     )
#     p3 = data_difference.plot(
#         ax=ax3,
#         x="TLONG",
#         y="TLAT",
#         transform=tran,
#         cmap=cmocean.cm.balance,
#         vmin=-absmaxval_forcolorbar,
#         vmax=absmaxval_forcolorbar,
#         add_colorbar=False
#     )
#
#     ax1.set_title("high res")
#     ax2.set_title(title + "\n\nlow res")
#     ax3.set_title("difference")
#
#     cbar1 = fig.colorbar(
#         p1,
#         ax=[ax1, ax2],
#         location="left",
#         orientation="vertical",
#         fraction=0.05,
#         pad=0.08,
#         shrink=0.6
#     )
#     cbar2 = fig.colorbar(
#         p3,
#         ax=ax3,
#         location="right",
#         orientation="vertical",
#         fraction=0.05,
#         pad=0.08,
#         shrink=0.6
#     )
#
#     return fig
#
#
#
# def plot_2D_SAIcomp(
#     data_SAI,
#     data_noSAI,
#     data_difference,
#     title,
#     minval_forcolorbar=data2D_min,
#     maxval_forcolorbar=data2D_max,
#     absmaxval_forcolorbar=diff_max,
#     lon_west=TLONG_min,
#     lon_east=TLONG_max,
#     lat_south=TLAT_min,
#     lat_north=TLAT_max,
# ):
#     fig = plt.figure(figsize=(5, 15))
#
#     proj = ccrs.Robinson(central_longitude=(lon_west + lon_east)/2)
#     tran = ccrs.PlateCarree()
#     ax1 = fig.add_subplot(3, 1, 1, projection=proj)
#     ax2 = fig.add_subplot(3, 1, 2, projection=proj)
#     ax3 = fig.add_subplot(3, 1, 3, projection=proj)
#
#     for ax in [ax1, ax2, ax3]:
#         # ax.gridlines()
#         ax.coastlines(resolution='110m', color='black')
#         ax.set_extent((lon_west, lon_east, lat_south, lat_north))
#         # if isinstance(proj, ccrs.Robinson):
#             # ax.set_facecolor("white")
#
#     p1 = data_SAI.plot(
#         ax=ax1,
#         x="TLONG",
#         y="TLAT",
#         transform=tran,
#         cmap=cmocean.cm.deep,
#         vmin=minval_forcolorbar,
#         vmax=maxval_forcolorbar,
#         add_colorbar=False
#     )
#     p2 = data_noSAI.plot(
#         ax=ax2,
#         x="TLONG",
#         y="TLAT",
#         transform=tran,
#         cmap=cmocean.cm.deep,
#         vmin=minval_forcolorbar,
#         vmax=maxval_forcolorbar,
#         add_colorbar=False
#     )
#     p3 = data_difference.plot(
#         ax=ax3,
#         x="TLONG",
#         y="TLAT",
#         transform=tran,
#         cmap=cmocean.cm.balance,
#         vmin=-absmaxval_forcolorbar,
#         vmax=absmaxval_forcolorbar,
#         add_colorbar=False
#     )
#
#     ax1.set_title(title, fontsize = 14)
#     fig.text(0.04, 0.75, "SAI", rotation=90,
#          va='center', ha='center', fontsize=16)
#     fig.text(0.04, 0.50, "no SAI", rotation=90,
#          va='center', ha='center', fontsize=16)
#     fig.text(0.04, 0.2, "difference", rotation=90,
#          va='center', ha='center', fontsize=16)
#
#     cbar1 = fig.colorbar(
#         p1,
#         ax=[ax1, ax2],
#         # location="left",
#         orientation="horizontal",
#         fraction=0.05,
#         pad=0.08,
#         # shrink=0.6
#     )
#     cbar2 = fig.colorbar(
#         p3,
#         ax=ax3,
#         # location="right",
#         orientation="horizontal",
#         fraction=0.05,
#         pad=0.08,
#         # shrink=0.6
#     )
#
#     return fig

In [ ]:
def plot_2D_anomaly(
        data_reference,
        data_SAI_endofcent,
        data_noSAI_endofcent,
        data_SAI,
        data_noSAI,
        title,
        minval_forcolorbar_ref=data2D_min,
        maxval_forcolorbar_ref=data2D_max,
        minval_forcolorbar_endofcent=data2D_min,
        maxval_forcolorbar_endofcent=data2D_max,
        absmaxval_forcolorbar_diff=diff_max,
        lon_west=TLONG_min,
        lon_east=TLONG_max,
        lat_south=TLAT_min,
        lat_north=TLAT_max,
):
    if var == "HMXL":
        cmap_for_refs = cmocean.cm.deep
    elif var == "TREFHT":
        cmap_for_refs = cmocean.cm.balance
    elif var == "PRECIP":
        cmap_for_refs = cmocean.cm.rain

    cbar_pad = 0.02
    cbar_shrink = 1.00
    cbar_orientation = "horizontal"
    cbar_extend = "neither"

    data_difference_SAI_noSAI = data_SAI - data_noSAI
    data_difference_noSAI_ref = data_noSAI - data_reference
    data_difference_SAI_ref = data_SAI - data_reference

    if module == "ocn":
        data_difference_SAI_noSAI = data_difference_SAI_noSAI.assign_coords(TLONG=data_SAI.TLONG)
        data_difference_SAI_noSAI = data_difference_SAI_noSAI.assign_coords(TLAT=data_SAI.TLAT)
        data_difference_noSAI_ref = data_difference_noSAI_ref.assign_coords(TLONG=data_noSAI.TLONG)
        data_difference_noSAI_ref = data_difference_noSAI_ref.assign_coords(TLAT=data_noSAI.TLAT)
        data_difference_SAI_ref = data_difference_SAI_ref.assign_coords(TLONG=data_SAI.TLONG)
        data_difference_SAI_ref = data_difference_SAI_ref.assign_coords(TLAT=data_SAI.TLAT)
        x_forplots = "TLONG"
        y_forplots = "TLAT"
    else:
        data_difference_SAI_noSAI = data_difference_SAI_noSAI.assign_coords(lon=data_SAI.lon)
        data_difference_SAI_noSAI = data_difference_SAI_noSAI.assign_coords(lat=data_SAI.lat)
        data_difference_noSAI_ref = data_difference_noSAI_ref.assign_coords(lon=data_noSAI.lon)
        data_difference_noSAI_ref = data_difference_noSAI_ref.assign_coords(lat=data_noSAI.lat)
        data_difference_SAI_ref = data_difference_SAI_ref.assign_coords(lon=data_SAI.lon)
        data_difference_SAI_ref = data_difference_SAI_ref.assign_coords(lat=data_SAI.lat)
        x_forplots = "lon"
        y_forplots = "lat"

    fig = plt.figure(figsize=(10, 12), constrained_layout=True)

    proj = ccrs.Robinson(central_longitude=(lon_west + lon_east) / 2)
    tran = ccrs.PlateCarree()
    ax11 = fig.add_subplot(3, 2, 1, projection=proj)
    ax12 = fig.add_subplot(3, 2, 2, projection=proj)
    ax21 = fig.add_subplot(3, 2, 3, projection=proj)
    ax22 = fig.add_subplot(3, 2, 4, projection=proj)
    ax31 = fig.add_subplot(3, 2, 5, projection=proj)
    ax32 = fig.add_subplot(3, 2, 6, projection=proj)

    for ax in [ax11, ax12, ax21, ax22, ax31, ax32]:
        ax.coastlines(resolution='110m', color='black')
        ax.set_extent((lon_west, lon_east, lat_south, lat_north), crs=tran)
        ax.tick_params(axis='both', labelsize=16)

    p11 = data_reference.plot(
        ax=ax11,
        x=x_forplots,
        y=y_forplots,
        transform=tran,
        cmap=cmap_for_refs,
        vmin=minval_forcolorbar_ref,
        vmax=maxval_forcolorbar_ref,
        cbar_kwargs={
            "orientation": cbar_orientation,
            "pad": cbar_pad,  # space between plot and colorbar
            "shrink": cbar_shrink,  # fraction of axes width
            "label": "",
            "extend": cbar_extend,
        }
    )
    p21 = data_noSAI_endofcent.plot(
        ax=ax21,
        x=x_forplots,
        y=y_forplots,
        transform=tran,
        cmap=cmap_for_refs,
        vmin=minval_forcolorbar_endofcent,
        vmax=maxval_forcolorbar_endofcent,
        cbar_kwargs={
            "orientation": cbar_orientation,
            "pad": cbar_pad,
            "shrink": cbar_shrink,
            "label": "",
            "extend": cbar_extend,
        }
    )
    p31 = data_SAI_endofcent.plot(
        ax=ax31,
        x=x_forplots,
        y=y_forplots,
        transform=tran,
        cmap=cmap_for_refs,
        vmin=minval_forcolorbar_endofcent,
        vmax=maxval_forcolorbar_endofcent,
        cbar_kwargs={
            "orientation": cbar_orientation,
            "pad": cbar_pad,
            "shrink": cbar_shrink,
            "label": "",
            "extend": cbar_extend,
        }
    )
    p12 = data_difference_SAI_noSAI.plot(
        ax=ax12,
        x=x_forplots,
        y=y_forplots,
        transform=tran,
        cmap=cmocean.cm.balance,
        vmin=-absmaxval_forcolorbar_diff,
        vmax=absmaxval_forcolorbar_diff,
        cbar_kwargs={
            "orientation": cbar_orientation,
            "pad": cbar_pad,
            "shrink": cbar_shrink,
            "label": "",
            "extend": cbar_extend,
        }
    )
    p22 = data_difference_noSAI_ref.plot(
        ax=ax22,
        x=x_forplots,
        y=y_forplots,
        transform=tran,
        cmap=cmocean.cm.balance,
        vmin=-absmaxval_forcolorbar_diff,
        vmax=absmaxval_forcolorbar_diff,
        cbar_kwargs={
            "orientation": cbar_orientation,
            "pad": cbar_pad,
            "shrink": cbar_shrink,
            "label": "",
            "extend": cbar_extend,
        }
    )
    p32 = data_difference_SAI_ref.plot(
        ax=ax32,
        x=x_forplots,
        y=y_forplots,
        transform=tran,
        cmap=cmocean.cm.balance,
        vmin=-absmaxval_forcolorbar_diff,
        vmax=absmaxval_forcolorbar_diff,
        cbar_kwargs={
            "orientation": cbar_orientation,
            "pad": cbar_pad,
            "shrink": cbar_shrink,
            "label": "",
            "extend": cbar_extend,
        }
    )

    ax11.set_title("Ref (no SAI 2000-2009)", fontsize=16)
    ax21.set_title("no SAI 2090-2098", fontsize=16)
    ax31.set_title("SAI 2090-2098", fontsize=16)
    ax12.set_title("SAI - no SAI", fontsize=16)
    ax22.set_title("no SAI - Ref", fontsize=16)
    ax32.set_title("SAI - Ref", fontsize=16)
    fig.suptitle(title, fontsize=18)

    for p in [p11, p21, p31, p12, p22, p32]:
        p.colorbar.ax.tick_params(labelsize=16)

    # fig.subplots_adjust(
    #     wspace=0.05,    # space between columns
    #     hspace=0.05,    # space between rows
    #     top=0.92        # leaves room for suptitle; lower = title closer to plots
    # )

    return fig



def plot_2D_resolandSAIcomp_3x3(
    data_highres_SAI,
    data_lowres_SAI,
    data_highres_noSAI,
    data_lowres_noSAI,
    data_difference_SAI,
    data_difference_noSAI,
    data_difference_highres,
    data_difference_lowres,
    title,
    minval_forcolorbar=data2D_min,
    maxval_forcolorbar=data2D_max,
    absmaxval_forcolorbar=diff_max
):
    """
    The suffixes of data_difference_ indicate which was left constant when determining the difference,
    e.g. data_difference_SAI = data_highres_SAI - data_lowres_SAI.
    """

    fig = plt.figure(figsize=(15, 12))

    fig.subplots_adjust(
        hspace=-0.5,
        wspace=0.05,
        bottom=0.12,    # space at bottom for the colorbars
    )

    proj = ccrs.Robinson(central_longitude=(TLONG_min + TLONG_max)/2)
    tran = ccrs.PlateCarree()
    ax11 = fig.add_subplot(3, 3, 1, projection=proj)
    ax12 = fig.add_subplot(3, 3, 2, projection=proj)
    ax13 = fig.add_subplot(3, 3, 3, projection=proj)
    ax21 = fig.add_subplot(3, 3, 4, projection=proj)
    ax22 = fig.add_subplot(3, 3, 5, projection=proj)
    ax23 = fig.add_subplot(3, 3, 6, projection=proj)
    ax31 = fig.add_subplot(3, 3, 7, projection=proj)
    ax32 = fig.add_subplot(3, 3, 8, projection=proj)

    for ax in [ax11, ax12, ax13, ax21, ax22, ax23, ax31, ax32]:
        # ax.gridlines()
        ax.coastlines(resolution='110m', color='black')
        ax.set_extent((lon_west, lon_east, lat_south, lat_north))
        # if isinstance(proj, ccrs.Robinson):
        #     ax.set_facecolor("white")

    p11 = data_highres_SAI.plot(
        ax=ax11,
        x="TLONG",
        y="TLAT",
        transform=tran,
        cmap=cmocean.cm.deep,
        vmin=minval_forcolorbar,
        vmax=maxval_forcolorbar,
        add_colorbar=False
    )
    p12 = data_lowres_SAI.plot(
        ax=ax12,
        x="TLONG",
        y="TLAT",
        transform=tran,
        cmap=cmocean.cm.deep,
        vmin=minval_forcolorbar,
        vmax=maxval_forcolorbar,
        add_colorbar=False
    )
    p13 = data_difference_SAI.plot(
        ax=ax13,
        x="TLONG",
        y="TLAT",
        transform=tran,
        cmap=cmocean.cm.balance,
        vmin=-absmaxval_forcolorbar,
        vmax=absmaxval_forcolorbar,
        add_colorbar=False
    )
    p21 = data_highres_noSAI.plot(
        ax=ax21,
        x="TLONG",
        y="TLAT",
        transform=tran,
        cmap=cmocean.cm.deep,
        vmin=minval_forcolorbar,
        vmax=maxval_forcolorbar,
        add_colorbar=False,
    )
    p22 = data_lowres_noSAI.plot(
        ax=ax22,
        x="TLONG",
        y="TLAT",
        transform=tran,
        cmap=cmocean.cm.deep,
        vmin=minval_forcolorbar,
        vmax=maxval_forcolorbar,
        add_colorbar=False,
    )
    p23 = data_difference_noSAI.plot(
        ax=ax23,
        x="TLONG",
        y="TLAT",
        transform=tran,
        cmap=cmocean.cm.balance,
        vmin=-absmaxval_forcolorbar,
        vmax=absmaxval_forcolorbar,
        add_colorbar=False,
    )
    p31 = data_difference_highres.plot(
        ax=ax31,
        x="TLONG",
        y="TLAT",
        transform=tran,
        cmap=cmocean.cm.balance,
        vmin=-absmaxval_forcolorbar,
        vmax=absmaxval_forcolorbar,
        add_colorbar=False,
    )
    p32 = data_difference_lowres.plot(
        ax=ax32,
        x="TLONG",
        y="TLAT",
        transform=tran,
        cmap=cmocean.cm.balance,
        vmin=-absmaxval_forcolorbar,
        vmax=absmaxval_forcolorbar,
        add_colorbar=False,
    )

    fig.suptitle(title, fontsize=18)
    ax11.set_title("high res", fontsize=16)
    ax12.set_title("low res", fontsize=16)
    ax13.set_title("difference", fontsize=16)
    fig.text(0.04, 0.75, "SAI", rotation=90,
         va='center', ha='center', fontsize=16)
    fig.text(0.04, 0.50, "no SAI", rotation=90,
         va='center', ha='center', fontsize=16)
    fig.text(0.04, 0.25, "difference", rotation=90,
         va='center', ha='center', fontsize=16)
    for ax in [ax21, ax22, ax23, ax31, ax32]:
        ax.set_title("")

    # Upper colorbar for the data values
    cax1 = fig.add_axes([0.125, 0.1, 0.775, 0.02])
    cbar1 = fig.colorbar(p13, cax=cax1, orientation="horizontal")
    # cbar1.set_label("blabla", fontsize=12)
    cbar1.ax.tick_params(labelsize=10)

    # Lower cbar for the differences
    cax2 = fig.add_axes([0.125, 0.05, 0.775, 0.02])
    cbar2 = fig.colorbar(p11, cax=cax2, orientation="horizontal")
    cbar2.ax.tick_params(labelsize=10)

    return fig

In [ ]:
# ### Plotting raw data in March and April



# # data_hr_time_selected = data_hr_SAI.sel(time = (data_hr_SAI.time.dt.month.isin([4,5])) & (data_hr_SAI.time.dt.year >= 2086))
# data_hr_time_selected = data_hr.sel(time = (data_hr.time.dt.month.isin([5])) & (data_hr.time.dt.year >= 2050))


# for t in data_hr_time_selected.time:
#     time_str = str(t.values.item())[0:10] # to get e.g. '2007-01-01' instead of '2007-01-01 00:00:00'

# #     fig = plot_2D_resolcomp(
# #         data_hr_SAI.sel(time=time_str),
# #         data_lr_SAI.sel(time=time_str),
# #         data_diff_SAI.sel(time=time_str),
# #         title = var + unit + f" with SAI at {time_str}"
# #     )

#     fig = plot_2D_resolandSAIcomp_3x3(
#         data_hr_SAI.sel(time=time_str),
#         data_lr_SAI.sel(time=time_str),
#         data_hr.sel(time=time_str),
#         data_lr.sel(time=time_str),
#         data_diff_SAI.sel(time=time_str),
#         data_diff_noSAI.sel(time=time_str),
#         data_diff_hr.sel(time=time_str),
#         data_diff_lr.sel(time=time_str),
#         title = var + unit + f" comparison at {time_str}",
#     )

#     print(time_str)
#     save_path = Path.cwd() / "plots"
# #     file_name = var + "_SAI_resolcomp_" + time_str
#     file_name = var + "_3x3_SAIresolcomp_" + time_str
#     fig.savefig(save_path/file_name) ### WARNING: pdf takes several minutes per figure (per month)
#     plt.close(fig)                   ### png takes about a minute

In [ ]:
# ### OLD VERSION OF THE BLOCK BELOW
# ### Plotting 10-year running mean every 5 years
# ### WARNING: the end of this range is too large, hence the break statement in the end; also, the last mean doesn't have the same range as the others

# for t_increment in range(0, 20): 
#     start_time = data_hr.time.dt.year.min().values.item() + t_increment * 5
#     end_time = min(start_time + 9, data_hr.time.dt.year.max().values.item())
#     start_time_str = str(start_time)
#     end_time_str = str(end_time)
    
#     data_hr_slice = data_hr.sel(time=slice(start_time_str, end_time_str))
#     data_lr_slice = data_lr.sel(time=slice(start_time_str, end_time_str))
#     data_diff_slice = data_diff.sel(time=slice(start_time_str, end_time_str))
    
#     fig = plot_2D_resolcomp(
#         data_hr_slice.mean(dim="time").compute(),
#         data_lr_slice.mean(dim="time").compute(),
#         data_diff_slice.mean(dim="time").compute(),
#         title="Mean " + var + unit + " for " + start_time_str + " - " + end_time_str,
#         maxval_forcolorbar=450,
#         absmaxval_forcolorbar=450
#     )
#     file_name = var + "_10yrmean_resolcomp_beginning" + start_time_str
#     print(file_name)
#     save_path = Path.cwd() / "plots_runningmean"
#     fig.savefig(save_path / file_name)
#     plt.close(fig)
    
#     del data_hr_slice, data_lr_slice, data_diff_slice
    
#     if end_time >= data_hr.time.dt.year.max().values.item():
#         break

In [ ]:
### User input
years_rollingmean = 10
start_year = 2050

############

unit = vars_dict[var]['unit']
end_year = min(start_year + years_rollingmean - 1, 2098)
start_time_str = str(start_year)
end_time_str = str(end_year)

# data_all = [data_hr_SAI, data_lr_SAI, data_hr, data_lr, data_diff_SAI, data_diff_noSAI, data_diff_hr, data_diff_lr]
data_all = [data_lr_SAI, data_lr]

data_rolling = []
for i in range(len(data_all)):
    data_rolling.append(data_all[i].rolling(time=years_rollingmean, center=True).mean())

############

# fig = plot_2D_resolcomp(
#     data_rolling[0], # data_hr_SAI sliced
#     data_rolling[1], # data_lr_SAI sliced
#     data_rolling[4], # data_diff_SAI sliced
#     title= "Mean " + var + unit + " with SAI for " + start_time_str + " - " + end_time_str,
#     maxval_forcolorbar=data2D_max,
#     absmaxval_forcolorbar=diff_absmax
# )
# file_name = var + "_resolcomp_" + str(years_rollingmean) + "yrmean_resolcomp_hr_beginning" + start_time_str
# print(file_name)
# save_path = Path.cwd() / "plots"
# fig.savefig(save_path / file_name)
# plt.close(fig)


# fig = plot_2D_SAIcomp(
#     data_rolling[0], # data_hr_SAI sliced
#     data_rolling[2], # data_hr sliced
#     data_rolling[6], # data_diff_hr sliced
#     title="Mean " + var + unit + " for " + start_time_str + " - " + end_time_str,
#     minval_forcolorbar=-1500,
#     maxval_forcolorbar=1500, #data2D_max,
#     absmaxval_forcolorbar=750, #1500 #diff_absmax
# )
# file_name = var + "_SAI_" + str(years_rollingmean) + "yrmean_resolcomp_hr_beginning" + start_time_str
# print(file_name)
# save_path = Path.cwd() / "plots"
# fig.savefig(save_path / file_name)
# plt.close(fig)


# fig = plot_2D_resolandSAIcomp_3x3(
#     *data_rolling,
#     title = "Mean " + var + unit + " for " + start_time_str + " - " + end_time_str,
#     minval_forcolorbar = data2D_min,
#     maxval_forcolorbar = data2D_max,
#     absmaxval_forcolorbar = diff_absmax
# )
# file_name = var + "_3x3_SAIresolcomp_" + str(years_rollingmean) + "yrmean_beginning" + start_time_str
# print(file_name)
# save_path = Path.cwd() / "plots"
# fig.savefig(save_path / file_name)
# plt.close(fig)


# fig = plot_2D_anomaly(
#     data_hr.sel(time=data_hr.time.dt.year.isin(range(2000, 2010))).mean(dim="time").compute(),
#     data_hr_SAI.sel(time=data_hr_SAI.time.dt.year.isin(range(2090, 2099))).mean(dim="time").compute(),
#     data_hr.sel(time=data_hr.time.dt.year.isin(range(2090, 2099))).mean(dim="time").compute(),
#     data_rolling[0], # data_hr_SAI sliced
#     data_rolling[2], # data_hr sliced
#     title = var + " anomaly " + unit + " for " + start_time_str + " - " + end_time_str,
#     absmaxval_forcolorbar_diff = 1500
# )
# file_name = var + "_hr_anomaly_" + str(years_rollingmean) + "yrmean_beginning" + start_time_str
# print(file_name)
# save_path = Path.cwd() / "plots"
# fig.savefig(save_path / file_name)
# plt.close(fig)



time_start = time.perf_counter()

for start_year in range(2055,2091,5):
    end_year = min(start_year + years_rollingmean - 1, 2098)
    start_time_str = str(start_year)
    end_time_str = str(end_year)
    
    fig = plot_2D_anomaly(
        data_lr_SAI.sel(time=data_lr_SAI.time.dt.year.isin(range(2000, 2010))).mean(dim="time").compute(),
        data_lr_SAI.sel(time=data_lr_SAI.time.dt.year.isin(range(2090, 2099))).mean(dim="time").compute(),
        data_lr.sel(time=data_lr.time.dt.year.isin(range(2090, 2099))).mean(dim="time").compute(),
        data_rolling[0].sel(time = data_rolling[0].time.dt.year == start_year).mean(dim="time"), # data_lr_SAI sliced
        data_rolling[1].sel(time = data_rolling[1].time.dt.year == start_year).mean(dim="time"), # data_lr sliced
        title = var + " anomaly " + unit + " for " + start_time_str + " - " + end_time_str,
        minval_forcolorbar_ref= -50,
        maxval_forcolorbar_ref= 50,
        minval_forcolorbar_endofcent= -50,
        maxval_forcolorbar_endofcent= 50,
        absmaxval_forcolorbar_diff = 15, # 2000
        lon_west  = -11.0,
        lon_east  = 32.0,
        lat_south = 36.0,
        lat_north = 72.0,
    )
    file_name = var + "_lr_anomaly_zoomed_" + str(years_rollingmean) + "yrmean_beginning" + start_time_str
    print(file_name)
    save_path = "/home/prouenhoff/ondemand/resol_comparison/" + var + "/plots/"  #Path.cwd() / "plots"
    fig.savefig(save_path + file_name)
    plt.close(fig)

print(time.perf_counter() - time_start)

In [ ]:
### User input
month_nr = 3
years_rollingmean = 10
start_year = 2050


month_names = ["ZEROETH MONTH (sth went wrong...)", "January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
month_name = month_names[month_nr]
unit = vars_dict[var]['unit']
end_year = start_year + years_rollingmean - 1
start_time_str = str(start_year)
end_time_str = str(end_year)

# data_all = [data_hr_SAI, data_lr_SAI, data_hr, data_lr, data_diff_SAI, data_diff_noSAI, data_diff_hr, data_diff_lr]
data_all = [data_lr_SAI, data_lr]

data_rolling = []
for i in range(len(data_all)):
    data_rolling.append(data_all[i].sel(time = data_all[i].time.dt.month == 1 + month_nr).rolling(year=years_rollingmean, center=True).mean())



# fig = plot_2D_resolcomp(
#     data_rolling[0], # data_hr_SAI sliced
#     data_rolling[1], # data_lr_SAI sliced
#     data_rolling[4], # data_diff_SAI sliced
#     title="Mean " + var + unit + " with SAI in " + month_name + " for " + start_time_str + " - " + end_time_str,
#     maxval_forcolorbar=data2D_max,
#     absmaxval_forcolorbar=diff_absmax
# )
# file_name = var + "_resolcomp_" + str(years_rollingmean) + "yrmean_" + month_name + "_resolcomp_hr_beginning" + start_time_str
# print(file_name)
# save_path = Path.cwd() / "plots"
# fig.savefig(save_path / file_name)
# plt.close(fig)


# fig = plot_2D_SAIcomp(
#     data_rolling[0], # data_hr_SAI sliced
#     data_rolling[2], # data_hr sliced
#     data_rolling[6], # data_diff_hr sliced
#     title="Mean " + var + unit + " in " + month_name + " for " + start_time_str + " - " + end_time_str,
#     minval_forcolorbar=-1500,
#     maxval_forcolorbar=1500, #data2D_max,
#     absmaxval_forcolorbar=750, #1500 #diff_absmax
# )
# file_name = var + "_SAI_" + str(years_rollingmean) + "yrmean_" + month_name + "_resolcomp_hr_beginning" + start_time_str
# print(file_name)
# save_path = Path.cwd() / "plots"
# fig.savefig(save_path / file_name)
# plt.close(fig)


# fig = plot_2D_resolandSAIcomp_3x3(
#     *data_rolling,
#     title = "Mean " + var + unit + " in " + month_name + " for " + start_time_str + " - " + end_time_str,
#     minval_forcolorbar = data2D_min,
#     maxval_forcolorbar = data2D_max,
#     absmaxval_forcolorbar = diff_absmax
# )
# file_name = var + "_3x3_SAIresolcomp_" + str(years_rollingmean) + "yrmean_" + month_name + "_beginning" + start_time_str
# print(file_name)
# save_path = Path.cwd() / "plots"
# fig.savefig(save_path / file_name)
# plt.close(fig)


# fig = plot_2D_anomaly(
#     data_hr.sel(time=(data_hr.time.dt.year.isin(range(2000, 2010))) & (data_hr.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
#     data_hr_SAI.sel(time=(data_hr_SAI.time.dt.year.isin(range(2090, 2099))) & (data_hr_SAI.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
#     data_hr.sel(time=(data_hr.time.dt.year.isin(range(2090, 2099))) & (data_hr.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
#     data_rolling[0], # data_hr_SAI sliced
#     data_rolling[2], # data_hr sliced
#     title = var + " anomaly " + unit + " in " + month_name + " for " + start_time_str + " - " + end_time_str,
#     absmaxval_forcolorbar_diff = 1500
# )
# file_name = var + "_hr_anomaly_" + str(years_rollingmean) + "yrmean_" + month_name + "_beginning" + start_time_str
# print(file_name)
# save_path = Path.cwd() / "plots"
# fig.savefig(save_path / file_name)
# plt.close(fig)


fig = plot_2D_anomaly(
    data_lr_SAI.sel(time=(data_lr_SAI.time.dt.year.isin(range(2000, 2010))) & (data_lr_SAI.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
    data_lr_SAI.sel(time=(data_lr_SAI.time.dt.year.isin(range(2090, 2099))) & (data_lr_SAI.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
    data_lr.sel(time=(data_lr.time.dt.year.isin(range(2090, 2099))) & (data_lr.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
    data_rolling[0], # data_lr_SAI sliced
    data_rolling[1], # data_lr sliced
    title = var + " anomaly " + unit + " in " + month_name + " for " + start_time_str + " - " + end_time_str,
    absmaxval_forcolorbar_diff = 3 # 2000
)
file_name = var + "_lr_anomaly_" + str(years_rollingmean) + "yrmean_" + month_name + "_beginning" + start_time_str
print(file_name)
save_path = Path.cwd() / "plots"
fig.savefig(save_path / file_name)
plt.close(fig)

In [ ]:
# ### OLDER VERSION OF BLOCK ABOVE
# ### User input
# month_nr = 3
#
#
# month_names = ["ZEROETH MONTH (sth went wrong...)", "January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
# month_name = month_names[month_nr]
# unit = vars_dict[var]['unit']
#
# # data_all = [data_hr_SAI, data_lr_SAI, data_hr, data_lr, data_diff_SAI, data_diff_noSAI, data_diff_hr, data_diff_lr]
# data_all = [data_lr_SAI, data_lr]
#
# print("Check range for t_increment!!\n")
#
# for t_increment in range(0, 20): ### WARNING: the end of this range is too large, hence the break statement in the end; also, the last mean doesn't have the same range as the others
#     start_time = data_hr_SAI.time.dt.year.min().values.item() + t_increment * 5
#     end_time = min(start_time + 10, data_hr_SAI.time.dt.year.max().values.item())
#     start_time_str = str(start_time)
#     end_time_str = str(end_time)
#
#     data_sliced = []
#     for i in range(len(data_all)):
#         data_sliced.append(data_all[i].sel(time=(data_all[i].time.dt.year.isin(range(start_time, end_time))) & (data_all[i].time.dt.month == 1 + month_nr)).mean(dim="time").compute())
#
#     # fig = plot_2D_resolcomp(
#     #     data_sliced[0], # data_hr_SAI sliced
#     #     data_sliced[1], # data_lr_SAI sliced
#     #     data_sliced[4], # data_diff_SAI sliced
#     #     title="Mean " + var + unit + " with SAI in " + month_name + " for " + start_time_str + " - " + end_time_str,
#     #     maxval_forcolorbar=data2D_max,
#     #     absmaxval_forcolorbar=diff_absmax
#     # )
#     # file_name = var + "_resolcomp_10yrmean_" + month_name + "_resolcomp_hr_beginning" + start_time_str
#     # print(file_name)
#     # save_path = Path.cwd() / "plots"
#     # fig.savefig(save_path / file_name)
#     # plt.close(fig)
#
#
#     # fig = plot_2D_SAIcomp(
#     #     data_sliced[0], # data_hr_SAI sliced
#     #     data_sliced[2], # data_hr sliced
#     #     data_sliced[6], # data_diff_hr sliced
#     #     title="Mean " + var + unit + " in " + month_name + " for " + start_time_str + " - " + end_time_str,
#     #     minval_forcolorbar=-1500,
#     #     maxval_forcolorbar=1500, #data2D_max,
#     #     absmaxval_forcolorbar=750, #1500 #diff_absmax
#     # )
#     # file_name = var + "_SAI_10yrmean_" + month_name + "_resolcomp_hr_beginning" + start_time_str
#     # print(file_name)
#     # save_path = Path.cwd() / "plots"
#     # fig.savefig(save_path / file_name)
#     # plt.close(fig)
#
#
#     # fig = plot_2D_resolandSAIcomp_3x3(
#     #     *data_sliced,
#     #     title = "Mean " + var + unit + " in " + month_name + " for " + start_time_str + " - " + end_time_str,
#     #     minval_forcolorbar = data2D_min,
#     #     maxval_forcolorbar = data2D_max,
#     #     absmaxval_forcolorbar = diff_absmax
#     # )
#     # file_name = var + "_3x3_SAIresolcomp_10yrmean_" + month_name + "_beginning" + start_time_str
#     # print(file_name)
#     # save_path = Path.cwd() / "plots"
#     # fig.savefig(save_path / file_name)
#     # plt.close(fig)
#
#
#     # fig = plot_2D_anomaly(
#     #     data_hr.sel(time=(data_hr.time.dt.year.isin(range(2000, 2010))) & (data_hr.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
#     #     data_hr_SAI.sel(time=(data_hr_SAI.time.dt.year.isin(range(2090, 2099))) & (data_hr_SAI.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
#     #     data_hr.sel(time=(data_hr.time.dt.year.isin(range(2090, 2099))) & (data_hr.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
#     #     data_sliced[0], # data_hr_SAI sliced
#     #     data_sliced[2], # data_hr sliced
#     #     title = var + " anomaly " + unit + " in " + month_name + " for " + start_time_str + " - " + end_time_str,
#     #     absmaxval_forcolorbar_diff = 1500
#     # )
#     # file_name = var + "_hr_anomaly_10yrmean_" + month_name + "_beginning" + start_time_str
#     # print(file_name)
#     # save_path = Path.cwd() / "plots"
#     # fig.savefig(save_path / file_name)
#     # plt.close(fig)
#
#
#     fig = plot_2D_anomaly(
#         data_lr_SAI.sel(time=(data_lr_SAI.time.dt.year.isin(range(2000, 2010))) & (data_lr_SAI.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
#         data_lr_SAI.sel(time=(data_lr_SAI.time.dt.year.isin(range(2090, 2099))) & (data_lr_SAI.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
#         data_lr.sel(time=(data_lr.time.dt.year.isin(range(2090, 2099))) & (data_lr.time.dt.month == 1 + month_nr)).mean(dim="time").compute(),
#         data_sliced[0], # data_lr_SAI sliced
#         data_sliced[1], # data_lr sliced
#         title = var + " anomaly " + unit + " in " + month_name + " for " + start_time_str + " - " + end_time_str,
#         absmaxval_forcolorbar_diff = 3 # 2000
#     )
#     file_name = var + "_lr_anomaly_10yrmean_" + month_name + "_beginning" + start_time_str
#     print(file_name)
#     save_path = Path.cwd() / "plots"
#     fig.savefig(save_path / file_name)
#     plt.close(fig)
#
#
#     print("\n")
#
#     if end_time >= data_hr_SAI.time.dt.year.max().values.item():
#         break
#
# print("Finished!")

### Create and visualize regional masks (HMXL only)

In [15]:
assert var == "HMXL"

In [28]:
TLAT_hr     = data_hr_SAI.sel(time="2060-01-01").TLAT
TLONG_hr    = (data_hr_SAI.sel(time="2060-01-01").TLONG + 180) % 360 -180
TLAT_lr     = data_lr.sel(time="2060-01-01").TLAT
TLONG_lr    = (data_lr.sel(time="2060-01-01").TLONG + 180) % 360 - 180

Labrador_condition_hr   = (TLONG_hr > -61) & (TLONG_hr < -45) & (TLAT_hr > 53) & (TLAT_hr < 68)
Irminger_condition_hr   = (TLONG_hr > -45) & (TLONG_hr < -19) & (TLAT_hr > 53) & (TLAT_hr < 66)
Westeurope_condition_hr = (TLONG_hr > -19) & (TLONG_hr < 1) & (TLAT_hr > 43) & (TLAT_hr < 66)
Nordic_condition_hr     = (TLONG_hr > -23) & (TLONG_hr < 20) & (TLAT_hr > 66) & (TLAT_hr < 85)

Labrador_condition_lr   = (TLONG_lr > -61) & (TLONG_lr < -45) & (TLAT_lr > 49) & (TLAT_lr < 68) \
    | (TLONG_lr >= -45) & (TLONG_lr < -36) & (TLAT_lr > 49) & (TLAT_lr < 60)
Irminger_condition_lr   = (TLONG_lr > -45) & (TLONG_lr < -36) & (TLAT_lr > 60) & (TLAT_lr < 66) \
    | (TLONG_lr >= -36) & (TLONG_lr < -19) & (TLAT_lr > 56) & (TLAT_lr < 66)
Westeurope_condition_lr = (TLONG_lr > -19) & (TLONG_lr < 1) & (TLAT_lr > 43) & (TLAT_lr < 66)  # Basically same as for hr
Nordic_condition_lr     = (TLONG_lr > -23) & (TLONG_lr < 20) & (TLAT_lr > 66) & (TLAT_lr < 85) # Basically same as for hr

hr_region_conditions = [Labrador_condition_hr, Irminger_condition_hr, Westeurope_condition_hr, Nordic_condition_hr]
lr_region_conditions = [Labrador_condition_lr, Irminger_condition_lr, Westeurope_condition_lr, Nordic_condition_lr]
region_names = ['Labrador', 'Irminger', 'Westeurope', 'Nordic']

In [17]:
bla = data_hr.where(data_hr.TLAT > 85).sel(time = data_hr.time.dt.year == 2000)
True in np.logical_not(np.isnan(bla.values))

True

In [33]:
datas_noSAI = [data_hr, data_lr]
datas_SAI   = [data_hr_SAI, data_lr_SAI]
datas_ref   = [data_hr, data_lr_SAI]
filenames   = ["regionmasks_hr.png", "regionmasks_lr.png"]
titles      = ["Regional masks used for HMXL [m] in high-res", "Regional masks used for HMXL [m] in low-res"]
colors      = ["tab:red", "tab:orange", "tab:green", "tab:blue"]
ax_titles   = ["Ref (2000-2009)", "no SAI 2090-2098", "SAI 2090-2098"]
TLONGS      = [TLONG_hr, TLONG_lr]
TLATS       = [TLAT_hr, TLAT_lr]
conditions  = [hr_region_conditions, lr_region_conditions]
month_nr    = 3


for resol_i in range(len(datas_noSAI)):
    fig = plt.figure(figsize=(15, 5), constrained_layout = True)
    proj = ccrs.Robinson(central_longitude=(TLONG_min + TLONG_max)/2)
    tran = ccrs.PlateCarree()

    # # # ax1 = fig.add_subplot(1, 1, 1, projection=proj)
    # # # p1 = datas_ref[resol_i].sel(time = datas_ref[resol_i].time.dt.year == 2000)\
    # # #         .mean(dim="time")\
    # # #         .plot(
    # # #             ax=ax1,
    # # #             x="TLONG",
    # # #             y="TLAT",
    # # #             transform=tran,
    # # #             cmap=cmocean.cm.deep,
    # # #             # rasterized=True,
    # # # )
    # # # ax1.coastlines(resolution='110m', color='black')
    # # # ax1.set_extent((TLONG_min, TLONG_max, TLAT_min, TLAT_max))
    
    ax1 = fig.add_subplot(1, 3, 1, projection=proj)
    ax2 = fig.add_subplot(1, 3, 2, projection=proj)
    ax3 = fig.add_subplot(1, 3, 3, projection=proj)
    axes_all = [ax1, ax2, ax3]

    p1 = datas_ref[resol_i].sel(time = (datas_ref[resol_i].time.dt.year.isin(range(2000, 2010))) & (datas_ref[resol_i].time.dt.month == 1 + month_nr))\
            .mean(dim="time")\
            .plot(
                ax=ax1,
                x="TLONG",
                y="TLAT",
                transform=tran,
                cmap=cmocean.cm.deep,
                vmin=0.0,
                vmax=data2D_max,
                add_colorbar=False,
                rasterized=True,
    )
    p2 = datas_noSAI[resol_i].sel(time = (datas_noSAI[resol_i].time.dt.year.isin(range(2090, 2099))) & (datas_noSAI[resol_i].time.dt.month == 1 + month_nr))\
            .mean(dim="time")\
            .plot(
                ax=ax2,
                x="TLONG",
                y="TLAT",
                transform=tran,
                cmap=cmocean.cm.deep,
                vmin=0.0,
                vmax=data2D_max,
                add_colorbar=False,
                rasterized=True,
    )
    p3 = datas_SAI[resol_i].sel(time = (datas_SAI[resol_i].time.dt.year.isin(range(2090, 2099))) & (datas_SAI[resol_i].time.dt.month == 1 + month_nr))\
            .mean(dim="time")\
            .plot(
                ax=ax3,
                x="TLONG",
                y="TLAT",
                transform=tran,
                cmap=cmocean.cm.deep,
                vmin=0.0,
                vmax=data2D_max,
                add_colorbar=False,
                rasterized=True,
    )

    for ax_i in range(len(axes_all)):
        axes_all[ax_i].set_title(ax_titles[ax_i])
        axes_all[ax_i].coastlines(resolution='110m', color='black')
        axes_all[ax_i].set_extent((TLONG_min, TLONG_max, TLAT_min, TLAT_max))
        for mask_i in range(len(hr_region_conditions)):
            axes_all[ax_i].contour(
                TLONGS[resol_i],
                TLATS[resol_i],
                conditions[resol_i][mask_i].astype(float),
                levels = [0.5],
                colors = colors[mask_i],
                transform=tran,
                linewidths=2,
                zorder=5,
    )

    cbar = fig.colorbar(
        p1,
        ax=[ax1, ax2, ax3],
        # location="left",
        orientation="horizontal",
        fraction=0.05,
        pad=0.08,
        shrink=1,
        label="HMXL [m]",
    )

    # fig.suptitle(titles[resol_i], fontsize=16)
    save_path = Path.cwd().parent / "HMXL/plots/regional_timeseries"
    fig.savefig(save_path / filenames[resol_i], dpi=300) #, bbox_inches="tight")
    plt.close(fig)

In [ ]:
### ### Visualize the masks for ref, SAI end of century, no SAI end of century
##
### month_names = ["ZEROETH MONTH (sth went wrong...)", "January", "February", "March", "April", "May", "June", "July", "August", "September", "October", "November", "December"]
##
### # Reference
### start_year  = 2000
### end_year    = 2009
### month_nr    = 3
##
### for region_i in range(len(hr_region_conditions)):
###     data_hr_masked = data_hr.where(hr_region_conditions[region_i])\
###         .assign_coords(TLONG=data_hr.TLONG, TLAT=data_hr.TLAT)\
###         .sel(time = (data_hr.time.dt.year.isin(range(start_year, end_year))) & (data_hr.time.dt.month == 1 + month_nr))\
###         .mean(dim="time")
###     data_lr_masked = data_lr_SAI.where(lr_region_conditions[region_i])\
###         .assign_coords(TLONG=data_lr.TLONG, TLAT=data_lr.TLAT)\
###         .sel(time = (data_lr_SAI.time.dt.year.isin(range(start_year, end_year))) & (data_lr_SAI.time.dt.month == 1 + month_nr))\
###         .mean(dim="time")
###     data_diff_masked = data_hr_masked * 0.0     # Dummy variable for plot funciton...
##
###     fig = plot_2D_resolcomp(
###         data_hr_masked,
###         data_lr_masked,
###         data_diff_masked,
###         title = f"{region_names[region_i]} mask, HMXL at {month_names[month_nr]} from {start_year} - {end_year}"
###     )
###     file_name = f"test_regionmask_{region_names[region_i]}_HMXL_ref"
###     save_path = Path.cwd() # / "plots_test"
###     fig.savefig(save_path / file_name)
###     plt.close(fig)
##
##
##
### # SAI end of century
### start_year  = 2090
### end_year    = 2098
##
### for region_i in range(len(hr_region_conditions)):
###     data_hr_masked = data_hr_SAI.where(hr_region_conditions[region_i])\
###         .assign_coords(TLONG=data_hr.TLONG, TLAT=data_hr.TLAT)\
###         .sel(time = (data_hr_SAI.time.dt.year.isin(range(start_year, end_year))) & (data_hr_SAI.time.dt.month == 1 + month_nr))\
###         .mean(dim="time")
###     data_lr_masked = data_lr_SAI.where(lr_region_conditions[region_i])\
###         .assign_coords(TLONG=data_lr.TLONG, TLAT=data_lr.TLAT)\
###         .sel(time = (data_lr_SAI.time.dt.year.isin(range(start_year, end_year))) & (data_lr_SAI.time.dt.month == 1 + month_nr))\
###         .mean(dim="time")
###     data_diff_masked = data_hr_masked * 0.0     # Dummy variable for plot funciton...
##
###     fig = plot_2D_resolcomp(
###         data_hr_masked,
###         data_lr_masked,
###         data_diff_masked,
###         title = f"{region_names[region_i]} mask, HMXL at {month_names[month_nr]} from {start_year} - {end_year} with SAI"
###     )
###     file_name = f"test_regionmask_{region_names[region_i]}_HMXL_SAI_endofcent"
###     save_path = Path.cwd() # / "plots_test"
###     fig.savefig(save_path / file_name)
###     plt.close(fig)
##
##
##
### # no SAI end of century
### for region_i in range(len(hr_region_conditions)):
###     data_hr_masked = data_hr.where(hr_region_conditions[region_i])\
###         .assign_coords(TLONG=data_hr.TLONG, TLAT=data_hr.TLAT)\
###         .sel(time = (data_hr.time.dt.year.isin(range(start_year, end_year))) & (data_hr.time.dt.month == 1 + month_nr))\
###         .mean(dim="time")
###     data_lr_masked = data_lr.where(lr_region_conditions[region_i])\
###         .assign_coords(TLONG=data_lr.TLONG, TLAT=data_lr.TLAT)\
###         .sel(time = (data_lr.time.dt.year.isin(range(start_year, end_year))) & (data_lr.time.dt.month == 1 + month_nr))\
###         .mean(dim="time")
###     data_diff_masked = data_hr_masked * 0.0     # Dummy variable for plot funciton...
##
###     fig = plot_2D_resolcomp(
###         data_hr_masked,
###         data_lr_masked,
###         data_diff_masked,
###         title = f"{region_names[region_i]} mask, HMXL at {month_names[month_nr]} from {start_year} - {end_year} without SAI"
###     )
###     file_name = f"test_regionmask_{region_names[region_i]}_HMXL_noSAI_endofcent"
###     save_path = Path.cwd() # / "plots_test"
###     fig.savefig(save_path / file_name)
###     plt.close(fig)

### Regional time series (HMXL only)

In [ ]:
# Get the TAREA in order to integrate HMXL over the surface below

area_hr_SAI = xr.open_dataset(file_list_hr_SAI[0], decode_timedelta = False)["TAREA"] * 1e-4 # cm² -> m²
area_lr_SAI = xr.open_dataset(file_list_lr_SAI[0], decode_timedelta = False)["TAREA"] * 1e-4 # cm² -> m²

area_hr_SAI = area_hr_SAI.roll(nlat = -nlat0, nlon = -nlon0, roll_coords = True).isel(nlat = slice(0, data_hr.nlat.size), nlon = slice(0, data_hr.nlon.size))

area_hr = area_hr_SAI  # The hr files do not contain that variable, but they're on the same grid
area_lr = area_lr_SAI  # Might as well do the same for area_lr...

In [ ]:
# Compute the annual maximum HMXL, as well as the surface integral of HMXL in April
# From both of these time series, we compute the running mean as well

month_nr = 3
years_rollingmean = 5

timeseries_max_hr = []  # timeseries_max_hr[0] = Labrador maximum HMXL time series, [1] = Irminger, ...
timeseries_max_lr = []
timeseries_max_hr_SAI = []
timeseries_max_lr_SAI = []

timeseries_sum_hr = []
timeseries_sum_lr = []
timeseries_sum_hr_SAI = []
timeseries_sum_lr_SAI = []

runningmean_max_hr = []
runningmean_max_lr = []
runningmean_max_hr_SAI = []
runningmean_max_lr_SAI = []

runningmean_sum_hr = []
runningmean_sum_lr = []
runningmean_sum_hr_SAI = []
runningmean_sum_lr_SAI = []

for region_i in range(len(hr_region_conditions)):
    print(region_i)

    # I really wish it weren't necessary to always assign these stupid coords again and again, but after all efforts, it is...
    masked_hr = data_hr.where(hr_region_conditions[region_i]).assign_coords(TLONG=data_hr.TLONG, TLAT=data_hr.TLAT)
    masked_lr = data_lr.where(lr_region_conditions[region_i]).assign_coords(TLONG=data_lr.TLONG, TLAT=data_lr.TLAT)
    masked_hr_SAI = data_hr_SAI.where(hr_region_conditions[region_i]).assign_coords(TLONG=data_hr.TLONG, TLAT=data_hr.TLAT)
    masked_lr_SAI = data_lr_SAI.where(lr_region_conditions[region_i]).assign_coords(TLONG=data_lr.TLONG, TLAT=data_lr.TLAT)

    # Compute time series of annual max HMXL
    timeseries_max_hr.append(masked_hr.groupby("time.year").max(dim=("time", "nlat", "nlon")))#.compute())
    timeseries_max_lr.append(masked_lr.groupby("time.year").max(dim=("time", "nlat", "nlon")))#.compute())
    timeseries_max_hr_SAI.append(masked_hr_SAI.groupby("time.year").max(dim=("time", "nlat", "nlon")))#.compute())
    timeseries_max_lr_SAI.append(masked_lr_SAI.groupby("time.year").max(dim=("time", "nlat", "nlon")))#.compute())

    print("    timeseries_max done")

    # Compute time series of annual March HMXL volume
    HMXLvolume_hr = masked_hr.sel(time=data_hr.time.dt.month==month_nr) * area_hr
    HMXLvolume_lr = masked_lr.sel(time=data_lr.time.dt.month==month_nr) * area_lr
    HMXLvolume_hr_SAI = masked_hr_SAI.sel(time=data_hr_SAI.time.dt.month==month_nr) * area_hr_SAI
    HMXLvolume_lr_SAI = masked_lr_SAI.sel(time=data_lr_SAI.time.dt.month==month_nr) * area_lr_SAI

    timeseries_sum_hr.append(HMXLvolume_hr.groupby("time.year").sum(dim=("time", "nlat", "nlon")).compute())
    timeseries_sum_lr.append(HMXLvolume_lr.groupby("time.year").sum(dim=("time", "nlat", "nlon")).compute())
    timeseries_sum_hr_SAI.append(HMXLvolume_hr_SAI.groupby("time.year").sum(dim=("time", "nlat", "nlon")).compute())
    timeseries_sum_lr_SAI.append(HMXLvolume_lr_SAI.groupby("time.year").sum(dim=("time", "nlat", "nlon")).compute())
    
    # Concatenate: until 2050, there is no SAI, so we can use the data from until then to extend the respective datasets for nicer plots.
    # For LR, the pre-SAI data is in the SAI data, and no-SAI starts at 2050 (coinciding with the start of SAI deployment).
    # We use just enough pre-SAI data to concatenate onto the no-SAI data for rolling means to exist at year 2050 for nice plots.
    years_needed_preSAI = years_rollingmean # np.ceil(1.5*years_rollingmean).astype(int)
    startyear_for_concat_lr = 2050 - years_needed_preSAI
    
    timeseries_max_lr[region_i] = xr.concat(
        [
            timeseries_max_lr_SAI[region_i]\
                .sel(year=timeseries_max_lr_SAI[region_i].year\
                    .isin(range(startyear_for_concat_lr, 2050))),
            timeseries_max_lr[region_i]
        ],
        dim = "year"
    )
    timeseries_sum_lr[region_i] = xr.concat(
        [
            timeseries_sum_lr_SAI[region_i]\
                .sel(year=timeseries_sum_lr_SAI[region_i].year\
                    .isin(range(startyear_for_concat_lr, 2050))),
            timeseries_sum_lr[region_i]
        ],
        dim = "year"
    )

    # Now for HR, the pre-SAI data is in the no-SAI data, and since the SAI-data starts at 2045 already, so we have to modify startyear_for_concat.
    startyear_for_concat_hr = 2045 - years_needed_preSAI
    
    timeseries_max_hr_SAI[region_i] = xr.concat(
        [
            timeseries_max_hr[region_i]\
                .sel(year=timeseries_max_hr[region_i].year\
                    .isin(range(startyear_for_concat_hr, 2045))),
            timeseries_max_hr_SAI[region_i]
        ],
        dim = "year"
    )
    timeseries_sum_hr_SAI[region_i] = xr.concat(
        [
            timeseries_sum_hr[region_i]\
                .sel(year=timeseries_sum_hr[region_i].year\
                    .isin(range(startyear_for_concat_hr, 2045))),
            timeseries_sum_hr_SAI[region_i]
        ],
        dim = "year"
    )

    print("    timeseries_sum done")

    # Compute running mean of annual max HMXL
    runningmean_max_hr.append(timeseries_max_hr[region_i].rolling(year=years_rollingmean, center=True).mean())#.values)
    runningmean_max_lr.append(timeseries_max_lr[region_i].rolling(year=years_rollingmean, center=True).mean())#.values)
    runningmean_max_hr_SAI.append(timeseries_max_hr_SAI[region_i].rolling(year=years_rollingmean, center=True).mean())#.values)
    runningmean_max_lr_SAI.append(timeseries_max_lr_SAI[region_i].rolling(year=years_rollingmean, center=True).mean())#.values)

    print("    runningmean_max done")

    # Compute running mean of annual March HMXL volume
    runningmean_sum_hr.append(timeseries_sum_hr[region_i].rolling(year=years_rollingmean, center=True).mean())#.values)
    runningmean_sum_lr.append(timeseries_sum_lr[region_i].rolling(year=years_rollingmean, center=True).mean())#.values)
    runningmean_sum_hr_SAI.append(timeseries_sum_hr_SAI[region_i].rolling(year=years_rollingmean, center=True).mean())#.values)
    runningmean_sum_lr_SAI.append(timeseries_sum_lr_SAI[region_i].rolling(year=years_rollingmean, center=True).mean())#.values)

    print("    runningmean_sum done")

    print("\n")

In [ ]:
timeseries_preSAI_all = [timeseries_max_hr, timeseries_max_lr_SAI, timeseries_sum_hr, timeseries_sum_lr_SAI]
timeseries_SAI_all  = [timeseries_max_hr_SAI, timeseries_max_lr_SAI, timeseries_sum_hr_SAI, timeseries_sum_lr_SAI]
timeseries_noSAI_all  = [timeseries_max_hr, timeseries_max_lr, timeseries_sum_hr, timeseries_sum_lr]
runningmean_preSAI_all = [runningmean_max_hr, runningmean_max_lr_SAI, runningmean_sum_hr, runningmean_sum_lr_SAI]
runningmean_SAI_all  = [runningmean_max_hr_SAI, runningmean_max_lr_SAI, runningmean_sum_hr_SAI, runningmean_sum_lr_SAI]
runningmean_noSAI_all  = [runningmean_max_hr, runningmean_max_lr, runningmean_sum_hr, runningmean_sum_lr]
titles = [
    "Max of HMXL [m] and 5-year running mean: high-res",
    "Max of HMXL [m] and 5-year running mean: low-res",
    "Surface integral of HMXL [m3] in March and 5-year running mean: high-res",
    "Surface integral of HMXL [m3] in March and 5-year running mean: low-res",
]
filenames = [
    "HMXL_regional_max_hr_SAIvsNOSAI.pdf",
    "HMXL_regional_max_lr_SAIvsNOSAI.pdf",
    "HMXL_regional_sum_hr_SAIvsNOSAI.pdf",
    "HMXL_regional_sum_lr_SAIvsNOSAI.pdf",
]
SAI_startyears   = [2045, 2050, 2045, 2050]

for i in range(len(timeseries_SAI_all)):   
    timeseries_preSAI   = timeseries_preSAI_all[i]
    timeseries_SAI      = timeseries_SAI_all[i]
    timeseries_noSAI    = timeseries_noSAI_all[i]
    runningmean_preSAI  = runningmean_preSAI_all[i]
    runningmean_SAI     = runningmean_SAI_all[i]
    runningmean_noSAI   = runningmean_noSAI_all[i]
    title       = titles[i]
    file_name   = filenames[i]
    SAI_startyear    = SAI_startyears[i]

    print("Starting with ", title)

    fig, ax = plt.subplots(figsize=(8,5))
    colors = ["tab:red", "tab:orange", "tab:green", "tab:blue"]

    for region_i in range(len(hr_region_conditions)):
        timeseries_preSAI_regional = timeseries_preSAI[region_i].sel(year=timeseries_preSAI[region_i].year <= SAI_startyear - 1) # .isin(range(2000, SAI_startyear+1)))
        timeseries_SAI_regional = timeseries_SAI[region_i].sel(year=timeseries_SAI[region_i].year >= SAI_startyear-1) #years_needed_preSAI)
        timeseries_noSAI_regional = timeseries_noSAI[region_i].sel(year=timeseries_noSAI[region_i].year >= SAI_startyear-1) #years_needed_preSAI)
        ax.plot(
            timeseries_preSAI_regional.year,
            timeseries_preSAI_regional,
            color = colors[region_i],
            alpha = 0.2,
            label = "_nolegend_",
        )
        ax.plot(
            timeseries_noSAI_regional.year,
            timeseries_noSAI_regional,
            color = colors[region_i],
            alpha = 0.2,
            label = "_nolegend_",
        )
        ax.plot(
            timeseries_SAI_regional.year,
            timeseries_SAI_regional,
            color = colors[region_i],
            alpha = 0.2,
            label = "_nolegend_",
            linestyle = "dashed"
        )

    for region_i in range(len(hr_region_conditions)):
        runningmean_preSAI_regional = runningmean_preSAI[region_i].sel(year=runningmean_preSAI[region_i].year <= np.ceil(SAI_startyear - 0.5*years_needed_preSAI).astype(int)) #.isin(range(2000, SAI_startyear+1)))
        runningmean_SAI_regional = runningmean_SAI[region_i].sel(year=runningmean_SAI[region_i].year >= SAI_startyear - years_needed_preSAI)
        runningmean_noSAI_regional = runningmean_noSAI[region_i].sel(year=runningmean_noSAI[region_i].year >= SAI_startyear - years_needed_preSAI)
        ax.plot(
            runningmean_preSAI_regional.year,
            runningmean_preSAI_regional,
            color = colors[region_i],
            label = region_names[region_i],
        )
        ax.plot(
            runningmean_noSAI_regional.year,
            runningmean_noSAI_regional,
            color = colors[region_i],
            label = "_nolegend", # region_names[region_i],
        )
        ax.plot(
            runningmean_SAI_regional.year,
            runningmean_SAI_regional,
            color = colors[region_i],
            label = "_nolegend_", # region_names[region_i] + ", SAI",
            linestyle = "dashed"
        )

    ax.set_title(title, fontsize=12)
    ax.grid(True)
    ax.legend()

    save_path = Path.cwd() / "plots/regional_timeseries"
    fig.savefig(save_path / file_name)

    plt.close(fig)

# TREFHT

In [ ]:
def D_from_means(data_ref_mean, data_SAI_mean, data_noSAI_mean):
    return np.abs(data_SAI_mean - data_ref_mean) - np.abs(data_noSAI_mean - data_ref_mean)

def significance(data_ref_mean, data_SAI_eof, data_noSAI_eof, n_boot=1000, seed=0):
    D_mean = D_from_means(data_ref_mean, data_SAI_eof.mean(dim="year"), data_noSAI_eof.mean(dim="year"))
    n = data_SAI_eof.sizes["year"]
    # assert data_noSAI_eof.sizes["year"] == n

    rng = np.random.default_rng(seed)
    counts = rng.multinomial(n, np.full(n, 1 / n), size=n_boot)   # (n_boot, n): draws per year
    weights = xr.DataArray(counts / n, dims=("boot", "year"))

    D_boot = D_from_means(
        data_ref_mean,
        xr.dot(weights, data_SAI_eof, dim="year"),
        xr.dot(weights, data_noSAI_eof, dim="year")
    )
    sigma = D_boot.std("boot")

    return D_mean, sigma

def get_significance_masks(D_mean, sigma, k=1.0):
    mask_significant = (k * sigma < np.abs(D_mean))
    mask_moderate    = ((D_mean + k * sigma) < 0).where(mask_significant).fillna(False)
    mask_exacerbate  = ((D_mean - k * sigma) > 0).where(mask_significant).fillna(False)
    mask_insignificant = np.logical_not(mask_significant)

    return mask_moderate, mask_exacerbate, mask_insignificant

In [ ]:
data_ref_mean_lr = data_lr_SAI.sel(time = data_lr_SAI.time.dt.year <= 2009).mean(dim="time")
data_SAI_eof = data_lr_SAI.groupby("time.year").mean(dim="time")
data_SAI_eof = data_SAI_eof.sel(year = data_SAI_eof.year >= 2069)
data_noSAI_eof = data_lr.groupby("time.year").mean(dim="time")
data_noSAI_eof = data_noSAI_eof.sel(year = data_noSAI_eof.year >= 2070)

In [ ]:
D_mean, sigma = significance(data_ref_mean_lr, data_SAI_eof, data_noSAI_eof)
D_mean = D_mean.compute()
sigma = sigma.compute()

mask_moderate, mask_exacerbate, mask_insignificant = get_significance_masks(D_mean, sigma, k=2.0)
mask_moderate = mask_moderate.compute()
mask_exacerbate = mask_exacerbate.compute()
mask_insignificant = mask_insignificant.compute()

In [ ]:
def plot_TREFHT_significance(
    data_reference,
    data_SAI,
    data_noSAI,
    mask_moderate,
    mask_exacerbate,
    title,
    n_level_cmap_ref,
    n_level_cmap_diff,
    cbar_ticks_ref,
    cbar_ticks_diff,
    minval_forcolorbar_ref=data2D_min,
    maxval_forcolorbar_ref=data2D_max,
    absmaxval_forcolorbar_diff=diff_max,
    lon_west=TLONG_min,
    lon_east=TLONG_max,
    lat_south=TLAT_min,
    lat_north=TLAT_max,
):

    cbar_pad         = 0.05  # space between plot and colorbar, def: 0.15 horizontal, 0.05 vertical
    cbar_shrink_ref  = 1.00  # fraction of axes width, def: 1.0
    cbar_shrink_diff = 1.00  # fraction of axes width, def: 1.0
    cbar_frac        = 0.15  # fraction of original Axes to use for colorbar, def: 0.15
    cbar_extend_ref  = "neither"
    cbar_extend_diff = "neither"
    cbar_orientation = "horizontal"
    cbar_ticks_ref   = cbar_ticks_ref
    cbar_ticks_diff  = cbar_ticks_diff
    cmap_ref  = cmocean.cm.thermal
    cmap_diff = cmocean.cm.balance
    proj = ccrs.Robinson(central_longitude=(lon_west + lon_east)/2)
    tran = ccrs.PlateCarree()

    levels_ref  = np.linspace(minval_forcolorbar_ref, maxval_forcolorbar_ref, n_level_cmap_ref + 1)
    levels_diff = np.linspace(-absmaxval_forcolorbar_diff, absmaxval_forcolorbar_diff, n_level_cmap_diff + 1)

    data_difference_noSAI_ref = data_noSAI - data_reference
    data_difference_noSAI_ref = data_difference_noSAI_ref.assign_coords(lon=data_noSAI.lon)
    data_difference_noSAI_ref = data_difference_noSAI_ref.assign_coords(lat=data_noSAI.lat)

    data_difference_SAI_ref = data_SAI - data_reference
    data_difference_SAI_ref = data_difference_SAI_ref.assign_coords(lon=data_SAI.lon)
    data_difference_SAI_ref = data_difference_SAI_ref.assign_coords(lat=data_SAI.lat)

    fig = plt.figure(figsize=(12, 6.5)) #, constrained_layout=True)
    gs  = GridSpec(nrows=2, ncols=3, figure=fig)
    ax_ref            = fig.add_subplot(gs[0,0],  projection=proj)
    ax_diff_noSAI_ref = fig.add_subplot(gs[1,0],  projection=proj)
    ax_diff_SAI_ref   = fig.add_subplot(gs[:,1:], projection=proj)

    for ax in [ax_ref, ax_diff_noSAI_ref, ax_diff_SAI_ref]:
        ax.coastlines(resolution='110m', color='black')
        ax.set_extent((lon_west, lon_east, lat_south, lat_north), crs=tran)
        ax.tick_params(axis='both', labelsize=16)

    p_ref = data_reference.plot(
        ax=ax_ref,
        x="lon",
        y="lat",
        transform=tran,
        cmap=cmap_ref,
        levels=levels_ref,
        cbar_kwargs={
            "orientation": cbar_orientation,
            "pad": cbar_pad,
            "shrink": cbar_shrink_ref,
            "label": "",
            "extend": cbar_extend_ref,
            "fraction": cbar_frac,
            "ticks": cbar_ticks_ref,
        }
    )
    p_diff_noSAI_ref = data_difference_noSAI_ref.plot(
        ax=ax_diff_noSAI_ref,
        x="lon",
        y="lat",
        transform=tran,
        cmap=cmap_diff,
        levels=levels_diff,
        cbar_kwargs={
            "orientation": cbar_orientation,
            "pad": cbar_pad,
            "shrink": cbar_shrink_diff,
            "label": "",
            "extend": cbar_extend_diff,
            "fraction": cbar_frac,
            "ticks": cbar_ticks_diff,
        }
    )
    p_diff_SAI_ref = data_difference_SAI_ref.plot(
        ax=ax_diff_SAI_ref,
        x="lon",
        y="lat",
        transform=tran,
        cmap=cmap_diff,
        levels=levels_diff,
        cbar_kwargs={
            "orientation": cbar_orientation,
            "pad": cbar_pad,
            "shrink": cbar_shrink_diff,
            "label": "",
            "extend": cbar_extend_diff,
            "fraction": cbar_frac,
            "ticks": cbar_ticks_diff,
            "aspect": 40, # only for this plot because it's bigger
        }
    )
    for mask, ls in [(mask_moderate, "dashed"), (mask_exacerbate, "dotted")]:
        ax_diff_SAI_ref.contour(
            data_SAI.lon,
            data_SAI.lat,
            mask.astype(float),
            levels=[0.5],
            colors="olivedrab",
            linestyles=ls,
            linewidths=1.5,
            transform=tran,
        )

    ax_ref.text(-0.06, 0.5, "Reference", transform=ax_ref.transAxes,
                rotation=90, va="center", ha="center", fontsize=17)
    ax_diff_noSAI_ref.text(-0.06, 0.5, "no SAI - Ref", transform=ax_diff_noSAI_ref.transAxes,
                           rotation=90, va="center", ha="center", fontsize=17)
    ax_diff_SAI_ref.text(-0.025, 0.5, "SAI - Ref", transform=ax_diff_SAI_ref.transAxes,
                         rotation=90, va="center", ha="center", fontsize=17)
    # ax_diff_SAI_ref.set_title("Ref - SAI", fontsize=16)
    fig.suptitle(title, fontsize=18)

    for p in (p_ref, p_diff_noSAI_ref, p_diff_SAI_ref):
        p.colorbar.ax.tick_params(labelsize=14)
        # p.colorbar.set_label("TREFHT [°C]", fontsize=14)

    fig.subplots_adjust(
        wspace=0.15,    # space between columns
        hspace=0.01,    # space between rows
        top=0.97,       # leaves room for suptitle; higher -> plots closer to title
        left=0.05,      # space to the left
        right=0.98      # space to the right
    )

    ax_diff_SAI_ref.set_anchor('S')

    return fig



fig = plot_TREFHT_significance(
    data_lr_SAI.sel(time = data_lr_SAI.time.dt.year <= 2009).mean(dim="time"),
    data_lr_SAI.sel(time = data_lr_SAI.time.dt.year >= 2069).mean(dim="time"),
    data_lr.sel(time = data_lr.time.dt.year >= 2070).mean(dim="time"),
    mask_moderate,
    mask_exacerbate,
    "Mean TREFHT [°C]: Beginning VS End of Century",
    n_level_cmap_ref = 18,
    n_level_cmap_diff = 21,
    cbar_ticks_ref = np.arange(-32, 32+1, 8),
    cbar_ticks_diff = np.arange(-9, 9+1, 3),
    minval_forcolorbar_ref = -36,
    maxval_forcolorbar_ref = 36,
    absmaxval_forcolorbar_diff = 9,
)

save_path = Path.cwd() / "plots/regional_timeseries"
file_name = "TREFHT_lr.pdf"
fig.savefig(save_path / file_name)